## MASLD (Zhang et al., 2025)
**Reference**: Zhang et al. "Machine learning-based prediction of metabolic dysfunction-associated 
steatotic liver disease using NHANES data." PLOS ONE, 2025.

**Label**: CAP ≥ 302 dB/m (liver steatosis via VCTE), excluding hepatitis B/C and frequent alcohol

**Inclusion criteria**: Adults ≥20 years, NHANES 2017-2020 only (VCTE only available this cycle)

**Deviations from reference study**:
- No cycle deviation possible (CAP measurement only in 2017-2020)
- 5-fold stratified CV instead of train/test split

In [1]:
import pandas as pd
import numpy as np
import os, pickle

full = pd.read_pickle("/Users/alva/Documents/nhanes_tasks/data/NHANES/NHANES_master.pkl")

In [2]:
# --- MASLD task ---
# Reference: Zhang et al. (2025), PLOS ONE
# Population: Adults ≥20, NHANES 2017-2020
# Label: CAP ≥ 302 dB/m (liver ultrasound), excluding hepatitis B/C and frequent alcohol
# Deviation: Only 2017-2020 available (VCTE/CAP measurement); 5-fold stratified CV

subset_masld = full[
    (full['YEAR'] == '2017-2020') &
    (full['RIDAGEYR'] >= 20)
].copy()

subset_masld = subset_masld[
    (subset_masld['LBXHBS'] != 1) &
    (subset_masld['LBXHCR'] != 1)
]

subset_masld = subset_masld[
    ~(subset_masld['ALQ121'].isin([1, 2]))
]

subset_masld['MASLD'] = (subset_masld['LUXCAPM'] >= 302).astype(int)
subset_masld = subset_masld[subset_masld['LUXCAPM'].notna()]

features_masld = [
    'BMXWAIST', 'BMXBMI', 'LBXSTR', 'LBXSGL', 'LBXSATSI',
    'LBDHDD', 'LBXSCH', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH1',
    'LBXSAPSI', 'DIQ010', 'LBXSGTSI'
]

data_masld = subset_masld[features_masld + ['MASLD']].dropna()
X_masld = data_masld[features_masld].values
y_masld = data_masld['MASLD'].values

print(f"MASLD: n={len(y_masld)}, prevalence={y_masld.mean():.3f}, features={X_masld.shape[1]}")

MASLD: n=4847, prevalence=0.315, features=13


In [3]:
os.makedirs('/Users/alva/Documents/nhanes_tasks/data/tasks/', exist_ok=True)

task = {
    'X': X_masld,
    'y': y_masld,
    'features': features_masld,
    'name': 'masld'
}

with open('/Users/alva/Documents/nhanes_tasks/data/tasks/masld.pkl', 'wb') as f:
    pickle.dump(task, f)

print("Saved.")

Saved.
